In [1]:
import sys
# !{sys.executable} -m pip install xlrd

In [2]:
import pandas as pd
import os
import re

import arcpy
import pandas as pd
import geopandas as gpd

import json
# import asyncio
# import httpx


In [3]:
# name
project_name = 'korindo'

In [4]:
## arcgis project setup load
aprx = arcpy.mp.ArcGISProject('CURRENT')
print([m.name for m in aprx.listMaps()])

['gps', 'sra', 'korindo']


In [5]:
# setup map layer interaction arcpy
m = [m for m in aprx.listMaps() if m.name == project_name][0]

# check the list layers
print([l.name for l in m.listLayers()])

['ptsl_korindo', 'World Topographic Map', 'World Hillshade']


In [11]:
# the original workspace location
scratch_local_folder = r'\\Mac\Home\Documents\ArcGIS\Projects\treeo_analysis\treeo_analysis.gdb'
# arcpy.env.workspace = scratch_local_folder # later do this again if needed to list the result

#location of downloader webgis
workspace_downloader = r'C:\Users\q_bal\Documents\arcgis_restapi_notebooks\webgis_downloader'

In [12]:
# list data webgis MoF downloaded and converted to .gdb
mof_gdb_path = r'F:\gis_data\Timeseries.gdb'
arcpy.env.workspace = mof_gdb_path  # this should be overrided to scratch db again

arcpy.ListFeatureClasses()

['DEF_2003_2006',
 'DEF_2006_2009',
 'DEF_2009_2011',
 'DEF_2011_2012',
 'DEF_2012_2013',
 'DEF_2013_2014',
 'DEF_2014_2015',
 'DEF_2015_2016',
 'DEF_2016_2017',
 'DEF_2017_2018',
 'DEF_2018_2019',
 'DEF_2019_2020',
 'DEF_2020_2021',
 'DEF_2021_2022',
 'PL_1990',
 'PL_1996',
 'PL_2000',
 'PL_2003',
 'PL_2006',
 'PL_2009',
 'PL_2011',
 'PL_2012',
 'PL_2013',
 'PL_2014',
 'PL_2015',
 'PL_2016',
 'PL_2020',
 'PL_2021',
 'PL_2022',
 'REF_2017_2018',
 'REF_2018_2019',
 'REF_2019_2020',
 'REF_2020_2021',
 'REF_2021_2022',
 'PL_2017',
 'PL_2018',
 'PL_2019']

In [13]:
## now we are going to update the 2023 and 2024 data LC!
# redownload from webgis of MoF

# define AOI to download first, no need entire country to download
# this AOI will be converted to rectangle, bbox extent
# shapefile name for input, to get boundary aoi
aoi_bbox_to_download = r'\\Mac\Home\Documents\ArcGIS\Projects\treeo_analysis\Batas_PTSTL (1).gdb\Polygons'
geometry_type = 'POLYGON'

# description, properties of layer bbox
desc = arcpy.Describe(aoi_bbox_to_download)

# envelope bbox for arcgis rest api
envelope = desc.extent

print('envelope :',envelope)
xmin, ymin, xmax, ymax = envelope.XMin, envelope.YMin, envelope.XMax, envelope.YMax
print(xmax)

# make a new name json, for scrapy input
json_file_name = f'bbox_{project_name}.json'

# adding path and defined, use hardcoded instead
json_file_path = os.path.join(workspace_downloader,'input_json',json_file_name)

# implementation -> conversion to json for bbox
data_bbox = {}
data_bbox['xmin'] = xmin
data_bbox['ymin'] = ymin
data_bbox['xmax'] = xmax
data_bbox['ymax'] = ymax


# dictionary to json file
with open(json_file_path, 'w') as json_file:
    json.dump(data_bbox, json_file)

print(data_bbox)

## AFTER EDITED in spider, to put in spatial envelope, let's run the command scrapy and check
print(f'''
# RUN THIS COMMAND IN ROOT, NOT IN THIS JUPYTER, in the folder that has scrapy.cfg file
# scrapy crawl webgis_klhk_spider -O {json_file_name.replace(".json","")}_oid_list.json or you later change the naming
# {json_file_name.replace(".json","")}_oid_list.json contains objectids that needed for request
''')




envelope : 111.706493375 -0.457839252999975 112.109836017 -0.167195667999977 0 0 NaN NaN
112.10983601700002
{'xmin': 111.70649337500004, 'ymin': -0.45783925299997463, 'xmax': 112.10983601700002, 'ymax': -0.16719566799997665}

# RUN THIS COMMAND IN ROOT, NOT IN THIS JUPYTER, in the folder that has scrapy.cfg file
# scrapy crawl webgis_klhk_spider -O bbox_korindo_oid_list.json or you later change the naming
# bbox_korindo_oid_list.json contains objectids that needed for request



In [23]:
# json_file_path
# time_series_old = arcpy.ListFeatureClasses() # based on above env

In [24]:
# scratch_folder = r"\\Mac\\Home\\Documents\\ArcGIS\\Projects\\treeo_analysis\\"

In [32]:
### DO NOT DO IT AGAIN IF YOU ALREADY DONE THIS!
# list all the neccessary feature class
time_series_old = arcpy.ListFeatureClasses() # based on above env
# print([l for l in time_series_old])

#process the json collection based on the list data from the above (SCRAPY)
import os

# scratch folder output conversion json
# scratch_folder = r"\\Mac\Home\Documents\ArcGIS\Projects\treeo_analysis\"
scratch_folder = "//Mac/Home/Documents/ArcGIS/Projects/treeo_analysis/"

# Define the path to the root folder you want to search
# root_directory = r'C:\Users\q_bal\Documents\arcgis_restapi_notebooks\webgis_downloader\json_raw\'
root_directory = 'C:/Users/q_bal/Documents/arcgis_restapi_notebooks/webgis_downloader/json_raw/'

# Iterate through all directories and files recursively
list_output_lulc = []
for dirpath, dirnames, filenames in os.walk(root_directory):
    # dirpath is the path to the current directory
    # dirnames is a list of subdirectories in the current directory
    # filenames is a list of files in the current directory
    if 'output' in os.path.basename(dirpath):
        print('\n------------------------------------------------------')
        print(dirpath)

        folder_name = os.path.basename(dirpath)

        arcpy.management.CreateFileGDB(
            out_folder_path=scratch_folder, # put it in scratch folder
            out_name=folder_name, # make the same name
            out_version="CURRENT"
        )

        output_gdb = os.path.join(scratch_folder, folder_name+'.gdb')

        arcpy.env.addOutputsToMap = False
        # Iterate over the files in the current directory
        list_file = [] # initiate the empty list files every directory in dirpath (that filtered 'output')
        for filename in filenames:
            # Construct the full file path
            file_path = os.path.join(dirpath, filename)
    
            file_name = os.path.basename(file_path)

            output_file = os.path.join(output_gdb,file_name.replace('.json',''))
    
            if 'oid' in file_name:
                # Now you can process the file
                print(f"Found file: {file_path}")
                #list_file.append(file_path)
                arcpy.conversion.JSONToFeatures(
                    in_json_file=file_path,
                    out_features=output_file, # makesure, no extension for feature class in gdb
                    geometry_type="POLYGON" # lets hardcoded for now this part, later if we want to convert the line, point json, then revisit this to load json first
                )
                list_file.append(output_file)
                print(f'Done for {file_name}') 

        print('process to merge data')
        # to display the merge data
        arcpy.env.addOutputsToMap = True     
        output_file_merged = os.path.join(output_gdb,folder_name+'_merge')
        arcpy.management.Merge(
            inputs=list_file,
            output=output_file_merged,
            field_mappings=None,
            add_source="NO_SOURCE_INFO",
            field_match_mode="AUTOMATIC"
        )

        list_output_lulc.append(output_file_merged)

        print(f'all done for {folder_name}')


------------------------------------------------------
C:/Users/q_bal/Documents/arcgis_restapi_notebooks/webgis_downloader/json_raw/output_json_PL_2023
Found file: C:/Users/q_bal/Documents/arcgis_restapi_notebooks/webgis_downloader/json_raw/output_json_PL_2023\PL_2023_oid_279411.json
Done for PL_2023_oid_279411.json
Found file: C:/Users/q_bal/Documents/arcgis_restapi_notebooks/webgis_downloader/json_raw/output_json_PL_2023\PL_2023_oid_279426.json
Done for PL_2023_oid_279426.json
Found file: C:/Users/q_bal/Documents/arcgis_restapi_notebooks/webgis_downloader/json_raw/output_json_PL_2023\PL_2023_oid_279436.json
Done for PL_2023_oid_279436.json
Found file: C:/Users/q_bal/Documents/arcgis_restapi_notebooks/webgis_downloader/json_raw/output_json_PL_2023\PL_2023_oid_279452.json
Done for PL_2023_oid_279452.json
Found file: C:/Users/q_bal/Documents/arcgis_restapi_notebooks/webgis_downloader/json_raw/output_json_PL_2023\PL_2023_oid_279543.json
Done for PL_2023_oid_279543.json
Found file: C:/Us

In [33]:
# list_output_lulc = ['\\\\Mac\\\\Home\\\\Documents\\\\ArcGIS\\\\Projects\\\\treeo_analysis\\\\output_json_PL_2023.gdb\\\\output_json_PL_2023_merge',
#  '\\\\Mac\\\\Home\\\\Documents\\\\ArcGIS\\\\Projects\\\\treeo_analysis\\\\output_json_PL_2024.gdb\\\\output_json_PL_2024_merge']

In [34]:
prev_data_list = arcpy.ListFeatureClasses() # Timeseries (National data)
# list_output_lulc # is the one that just downloaded from json
prev_data_list = [os.path.join(mof_gdb_path, layername) for layername in prev_data_list]
all_list_data = prev_data_list + list_output_lulc

In [35]:
# all_list_data

In [36]:
# all_list_data[0].split('\\')[-1]
os.path.basename(all_list_data[-1])

'output_json_PL_2024_merge'

In [37]:
# for i in all_list_data:
#     print(i)

In [40]:
arcpy.env.overwriteOutput = True
### arcpy analysis in jupyter --> assuming the MoF data is downloaded
# based on the above printed result, the LU data is PL_*
dict_lc_fields = {
    'PL_1990': 'pl90_id',
    'PL_1996': 'pl96_id',
    'PL_2000': 'pl00_id',
    'PL_2003': 'pl03_id',
    'PL_2006': 'pl06_id',
    'PL_2009': 'pl09_id',
    'PL_2011': 'pl11_id',
    'PL_2012': 'pl12_id',
    'PL_2013': 'pl13_id',
    'PL_2014': 'pl14_id',
    'PL_2015': 'pl15_id',
    'PL_2016': 'pl16_id',
    'PL_2017': 'pl17_id',
    'PL_2018': 'PL_18_ID',
    'PL_2019': 'PL_19_R',
    'PL_2020': 'pl2020_id',
    'PL_2021': 'pl2021_id',
    'PL_2022': 'pl2022_id',
     'output_json_PL_2023_merge': 'PL2023_ID',
    'output_json_PL_2024_merge': 'PL2024_ID'
}

for i in all_list_data:
    file_name = os.path.basename(i)
    if 'PL_' in file_name: 
        print(f'processing {file_name}')
        arcpy.analysis.Clip(
            in_features=i,
            clip_features="ptsl_korindo",
            out_feature_class=os.path.join(scratch_local_folder,file_name+f'_Clipped_{project_name}'),
            cluster_tolerance=None
        )
        # print(i)

        # dissolving, to clean the data
        arcpy.management.Dissolve(
            in_features=os.path.join(scratch_local_folder,file_name+f'_Clipped_{project_name}'),
            out_feature_class=os.path.join(scratch_local_folder,file_name+f'_Clipped_{project_name}_dis'),
            dissolve_field=dict_lc_fields[file_name],
            statistics_fields=None,
            multi_part="MULTI_PART",
            unsplit_lines="DISSOLVE_LINES",
            concatenation_separator=""
        )
        
    else: # other
        print('other layer: ',file_name)
        

other layer:  DEF_2003_2006
other layer:  DEF_2006_2009
other layer:  DEF_2009_2011
other layer:  DEF_2011_2012
other layer:  DEF_2012_2013
other layer:  DEF_2013_2014
other layer:  DEF_2014_2015
other layer:  DEF_2015_2016
other layer:  DEF_2016_2017
other layer:  DEF_2017_2018
other layer:  DEF_2018_2019
other layer:  DEF_2019_2020
other layer:  DEF_2020_2021
other layer:  DEF_2021_2022
processing PL_1990
processing PL_1996
processing PL_2000
processing PL_2003
processing PL_2006
processing PL_2009
processing PL_2011
processing PL_2012
processing PL_2013
processing PL_2014
processing PL_2015
processing PL_2016
processing PL_2020
processing PL_2021
processing PL_2022
other layer:  REF_2017_2018
other layer:  REF_2018_2019
other layer:  REF_2019_2020
other layer:  REF_2020_2021
other layer:  REF_2021_2022
processing PL_2017
processing PL_2018
processing PL_2019
processing output_json_PL_2023_merge
processing output_json_PL_2024_merge


In [41]:
# IF ABOVE IS DONE, YOU CAN CONTINUE TO THIS!
# this layer list to include it in group layer
list_to_include_group = [lyr for lyr in m.listLayers() if f'Clipped_{project_name}' in lyr.name ]

group_layer = 'analysis_lulc'

m.createGroupLayer(group_layer)
print(f"Created new group layer: '{group_layer}'")

# --- Get the new group layer object from the map ---
# We find it by name. listLayers returns a list, so we take the first item [0].
group_layer = m.listLayers(group_layer)[0]

# --- Loop through the layers to be grouped and move them ---
for layer_name in list_to_include_group:
    # Find the layer object in the map by its name
    lyr_to_move = m.listLayers(layer_name)[0]
    
    # Use the map's dedicated function to move the layer into the group
    m.addLayerToGroup(group_layer, lyr_to_move)
    m.removeLayer(lyr_to_move)
    # print(f"Moved layer '{layer_name}' into group.")

# change back to default scratch folder
arcpy.env.workspace = scratch_local_folder

# this one for analysis later
list_layers_lc = [lyr for lyr in m.listLayers() if f'Clipped_{project_name}_dis' in lyr.name ]
print(sorted([lyr.name for lyr in list_layers_lc]))

# intersect all the data into single dataset
arcpy.analysis.Intersect(
    in_features=list_layers_lc,
    out_feature_class=fr"\\Mac\Home\Documents\ArcGIS\Projects\treeo_analysis\treeo_analysis.gdb\LC_MoF_1990_2024_{project_name}",
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT"
)

Created new group layer: 'analysis_lulc'
['PL_1990_Clipped_korindo_dis', 'PL_1996_Clipped_korindo_dis', 'PL_2000_Clipped_korindo_dis', 'PL_2003_Clipped_korindo_dis', 'PL_2006_Clipped_korindo_dis', 'PL_2009_Clipped_korindo_dis', 'PL_2011_Clipped_korindo_dis', 'PL_2012_Clipped_korindo_dis', 'PL_2013_Clipped_korindo_dis', 'PL_2014_Clipped_korindo_dis', 'PL_2015_Clipped_korindo_dis', 'PL_2016_Clipped_korindo_dis', 'PL_2017_Clipped_korindo_dis', 'PL_2018_Clipped_korindo_dis', 'PL_2019_Clipped_korindo_dis', 'PL_2020_Clipped_korindo_dis', 'PL_2021_Clipped_korindo_dis', 'PL_2022_Clipped_korindo_dis', 'output_json_PL_2023_merge_Clipped_korindo_dis', 'output_json_PL_2024_merge_Clipped_korindo_dis']


<Result '\\\\Mac\\Home\\Documents\\ArcGIS\\Projects\\treeo_analysis\\treeo_analysis.gdb\\LC_MoF_1990_2024_korindo'>

In [42]:
# dissolving, to clean the data
arcpy.management.Dissolve(
    in_features=f"LC_MoF_1990_2024_{project_name}",
    out_feature_class=f"LC_MoF_1990_2024_{project_name}_dissolved",
    dissolve_field=[field for layer,field in dict_lc_fields.items()],
    statistics_fields=None,
    multi_part="MULTI_PART",
    unsplit_lines="DISSOLVE_LINES",
    concatenation_separator=""
)

# calculate area_ha geodesic in the clean data
arcpy.management.CalculateGeometryAttributes(
    in_features=f"LC_MoF_1990_2024_{project_name}_dissolved",
    geometry_property="area_ha AREA_GEODESIC",
    length_unit="",
    area_unit="HECTARES",
    coordinate_system=None,
    coordinate_format="SAME_AS_INPUT"
)

<Result 'LC_MoF_1990_2024_korindo_dissolved'>

In [43]:
# select layer to re-clean the inconsistent (small) area_ha to eliminate (merge) to surround large feature polygon
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=f"LC_MoF_1990_2024_{project_name}_dissolved",
    selection_type="NEW_SELECTION",
    where_clause="area_ha <= 0.01",
    invert_where_clause=None
)

# cleaning implementation from previous step (select layer)
arcpy.management.Eliminate(
    in_features=f"LC_MoF_1990_2024_{project_name}_dissolved",
    out_feature_class=rf"\\Mac\Home\Documents\ArcGIS\Projects\treeo_analysis\treeo_analysis.gdb\LC_MoF_1990_2024_{project_name}_cleaned",
    selection="LENGTH",
    ex_where_clause="",
    ex_features=None
)

<Result '\\\\Mac\\Home\\Documents\\ArcGIS\\Projects\\treeo_analysis\\treeo_analysis.gdb\\LC_MoF_1990_2024_korindo_cleaned'>